In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import os
import numpy as np
from PIL import Image
from tqdm import tqdm
import torch
import torchvision.transforms as transforms

In [9]:
class Elastic_brain:
    def __init__(self, alpha=1.0, image_size=224):
        self.alpha = alpha
        self.image_size = image_size

        self.base_transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor()
        ])

        self.aug_transform = transforms.Compose([
            transforms.RandomResizedCrop(image_size, scale=(0.5, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor()
        ])

        self.to_pil = transforms.ToPILImage()

    def __call__(self, image):
        x1 = self.base_transform(image)
        x2 = self.aug_transform(image)
        lam = np.random.beta(self.alpha, self.alpha)
        mixed = lam * x1 + (1 - lam) * x2
        mixed = torch.clamp(mixed, 0.0, 1.0)
        return self.to_pil(mixed)

In [10]:
def salfmix_and_save(image_path, salfmix, save_dir):
    image = Image.open(image_path).convert("RGB")
    mixed_image = salfmix(image)
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, "Elastic_brain_" + os.path.basename(image_path))
    mixed_image.save(save_path)

In [11]:
def process_dataset(root_dir):

    salfmix = Elastic_brain(alpha=1.0, image_size=224)

    for cls in os.listdir(root_dir):

        cls_path = os.path.join(root_dir, cls)

        if not os.path.isdir(cls_path):
            continue

        print(f"Processing class: {cls}")

        for file in os.listdir(cls_path):

            if file.lower().endswith((".jpg", ".jpeg", ".png")):

                img_path = os.path.join(cls_path, file)

                salfmix_and_save(
                    img_path,
                    salfmix,
                    cls_path
                )

In [12]:
dataset_root = "/content/drive/MyDrive/database/eye/Elasic Training"
process_dataset(dataset_root)

Processing class: diabetic_retinopathy
Processing class: cataract
Processing class: glaucoma
Processing class: normal
